# Agentic Newsroom → Multi-Agent AI Systems with CrewAI & Gemini 2.5 Flash
---
<a href="https://ibb.co/PvLJjQt3"><img src="https://i.ibb.co/tTGnKX3k/research-agent.png" alt="research-agent" border="0"></a>

This system automates the creation of verified technical reports by using a three-agent pipeline to gather live web data, synthesize it into a structured document, and cross-reference the output against original sources to eliminate hallucinations.


In this notebook, we will build a **three-agent pipeline** that mirrors a real newsroom:

| Agent | Role | Responsibility |
|-------|------|----------------|
|  **Agent 1** | The Researcher | Uses live web search to gather raw information |
|  **Agent 2** | The Technical Writer | Synthesizes research into a structured report |
| **Agent 3** | The Fact-Checker | Audits the report for hallucinations & inaccuracies |



###  Key Concepts Covered
- **Hallucination Mitigation** — Why a Reviewer agent is essential in production
- **The Agentic Loop Problem** — What happens when an agent gets stuck
- **Token & Cost Management** — How to monitor your 'Token Burn'

#### **Architectural Diagram →** [Link](https://mermaid.ai/d/3990588c-7c74-467f-8963-628a3c70d872)

---

In [ ]:
from google.colab import userdata
userdata.get('secretName')

SecretNotFoundError: Secret secretName does not exist.



## Section 1: Installation

We install four packages:
- **`crewai`** — The multi-agent orchestration framework
- **`crewai-tools`** — Provides pre-built tools like the web search tool
- **`langchain-google-genai`** — LangChain's connector to Google's Gemini models
- **`google-generativeai`** — Google's core SDK (required by the above)

> **Expected install time:** ~2 minutes on a fresh Colab runtime.

In [ ]:
# ============================================================
# SECTION 1: INSTALLATION
# Run this cell first. It may take 1-2 minutes.
# The '-q' flag suppresses noisy output for cleaner display.
# ============================================================
!pip install -q crewai crewai-tools langchain-google-genai google-generativeai

## Building with CrewAI

Think of a standard Large Language Model (like ChatGPT or Gemini) as a highly capable but isolated freelancer.

**CrewAI** is a framework that allows you to take those freelancers and turn them into a structured company. Instead of relying on a single prompt-response interaction, you can assign roles, define responsibilities, provide tools, and orchestrate collaboration across multiple AI agents to complete complex, multi-step workflows.

---

## The 4 Core Building Blocks of CrewAI

To build our **Hallucination Mitigation System**, we only need to understand four foundational concepts. These ideas form the vocabulary we will use throughout our implementation.

### 1. Agents (The "Who")

In CrewAI, you don’t simply prompt an AI — you design a persona. Each **Agent** is defined with a clear role, a specific goal, and a backstory. This structure prevents the model from behaving like a generic chatbot and instead forces it to operate within a well-defined professional identity.

For example, our Fact-Checker agent can be designed to behave like a strict, detail-oriented auditor whose sole purpose is to challenge assumptions and verify claims.

### 2. Tasks (The "What")

A **Task** represents a clearly defined unit of work. Rather than vague instructions, each task includes a detailed description of what needs to be accomplished, what kind of output is expected (such as a structured Markdown report or JSON response), and which agent is responsible for completing it.

This clarity ensures accountability and keeps responsibilities sharply defined across the system.

### 3. Tools (The "Equipment")

By default, a language model only has access to its training data. CrewAI allows us to extend its capabilities by equipping agents with external tools.

For instance, our Researcher agent will be given access to the `SerperDevTool`, enabling it to browse the live internet and retrieve up-to-date information. This significantly reduces hallucination and improves factual grounding.

### 4. The Crew (The "Team")

The **Crew** ties everything together. It combines agents and tasks into an orchestrated workflow and determines how they interact.

In our case, we will use a **Sequential Process**, meaning the Researcher must complete fact-gathering before the Writer begins drafting. This enforces a controlled, reliable flow of information and ensures that writing is grounded in verified research.

---

### Why We Are Using CrewAI

When building real-world AI platforms, reliability and orchestration matter as much as model capability. CrewAI provides a structured way to manage multiple agents while maintaining control over how information flows between them.

One major advantage is its built-in **ReAct (Reason + Act) logic**, which encourages agents to think, evaluate available tools, take action, and reflect on their outputs before proceeding. This reasoning loop happens automatically under the hood.

CrewAI also enables a **zero-trust information architecture**. By isolating agents and limiting tool access, we can ensure that only the Researcher interacts with the web, while the Writer works strictly from verified inputs. This architectural separation dramatically reduces hallucination risks.

Finally, CrewAI is clean and Pythonic. Its readable structure makes it easy to map AI behavior directly to real-world workflow logic, making systems easier to debug, maintain, and scale.


## Section 2: API KEY CONFIGURATION

In [ ]:
# ============================================================
# SECTION 2: API KEY CONFIGURATION
# Safely loads secrets from Colab's Secret Manager.
# Falls back to os.environ for local development.
# ============================================================
import os

def load_secret(key_name: str) -> str:
    """
    Attempts to load a secret from Colab's userdata store.
    Falls back to environment variables for local development.
    Raises a clear error if the key is not found anywhere.
    """
    # --- Try Colab Secrets first (preferred in Colab) ---
    try:
        from google.colab import userdata
        value = userdata.get(key_name)
        if value:
            print(f" Loaded '{key_name}' from Colab Secrets.")
            return value
    except (ImportError, Exception):
        # Not running in Colab, or key not found in secrets
        pass

    # --- Fall back to environment variables ---
    value = os.environ.get(key_name)
    if value:
        print(f" Loaded '{key_name}' from os.environ.")
        return value

    # --- Key not found anywhere — raise a helpful error ---
    raise ValueError(
        f" API key '{key_name}' not found!\n"
        f"   → In Colab: Add it via the 🔑 Secrets panel (left sidebar).\n"
        f"   → Locally:  Run `export {key_name}='your_key_here'` in your terminal."
    )


# Load both required keys
GOOGLE_API_KEY = load_secret("GOOGLE_API_KEY")
SERPER_API_KEY = load_secret("SERPER_API_KEY")

# Set them as environment variables so LangChain/CrewAI can auto-discover them
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

print("\n🔐 All API keys configured successfully!")

---
## Section 3: Configuring the Language Model (Gemini 2.5 Flash)

We use **Gemini 2.5 Flash** — Google's fast, efficient model ideal for agentic loops.

### Why Flash and not Pro?
In agentic systems, the LLM is called **multiple times per task** (once per Thought/Action/Observation cycle). Using a heavier model like Gemini Pro would be:
- **Slower** — More latency per step
- **More expensive** — Tokens add up fast across many agent turns

Flash gives us ~80% of the capability at ~20% of the cost — a good production trade-off.

### Key Parameters Explained:
- **`temperature=0.3`** — Lower temperature = more deterministic, factual outputs. A value of `0` would be fully deterministic but less creative. We use `0.3` to allow some flexibility in writing style.
- **`max_tokens`** — Caps the response length per LLM call to control costs.

In [ ]:
# ============================================================
# SECTION 3: LLM CONFIGURATION
# We create ONE shared LLM instance and pass it to all agents.
# This is a best practice avoids re-initializing the model
# multiple times and makes it easy to swap models later.
# ============================================================
from langchain_google_genai import ChatGoogleGenerativeAI

# --- Initialize Gemini 2.5 Flash ---
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",       # Model identifier
    temperature=0.3,                  # Lower = more factual, less creative
    max_tokens=8192,                  # Max tokens per single LLM response
    google_api_key=GOOGLE_API_KEY,    # Explicitly passed (also set in env)
    # convert_system_message_to_human=True  # Uncomment if you hit role errors
)

print(" Gemini 2.5 Flash initialized successfully!")
print(f" Model: gemini-2.5-flash | Temperature: 0.3 | Max Tokens: 8192")

---
## Section 4: Tools → Giving Agents Superpowers

An LLM alone only has knowledge up to its training cutoff. **Tools** allow agents to:
- **Search the web** for real-time information (SerperDevTool)
- Read files, query databases, call APIs, and more



### How Tools Work in CrewAI's ReAct Loop:
```
THOUGHT:  "I need to find recent AI trends. I should search the web."
ACTION:   search_tool(query="latest AI trends 2025")
OBSERVATION: "[Search results returned...]"
THOUGHT:  "Based on the results, the top trends are..."
```
This **Thought → Action → Observation** cycle (called **ReAct**) is the core pattern of most modern AI agents.

>  **Note:** Only the Researcher agent gets the search tool. The Writer and Reviewer work only with text passed from the previous agent — this is intentional and mimics real-world role separation.

In [ ]:
# ============================================================
# SECTION 4: TOOLS
# SerperDevTool wraps the Serper.dev API, which provides structured Google search results in JSON format.
# ============================================================
from crewai_tools import SerperDevTool

# Initialize the search tool
# The tool automatically reads SERPER_API_KEY from os.environ
search_tool = SerperDevTool(
    n_results=5  # Fetch top 5 search results per query to balance info vs. tokens
)

print(" SerperDevTool initialized!")
print("   → This tool will be assigned exclusively to the Researcher agent.")
print("   → Each search call consumes ~1 Serper credit (2,500 free/month).")

---
## Section 5: Defining the Agents

Each `Agent` in CrewAI is defined by four key properties:

| Property | Purpose | Analogy |
|----------|---------|--------|
| **`role`** | The agent's job title | "Senior Researcher" |
| **`goal`** | What it's trying to achieve | Performance target |
| **`backstory`** | Personality & expertise context | Character background |
| **`tools`** | What capabilities it has access to | Toolbox |

### Why Does `backstory` Matter?
The backstory is **not just flavor text**. It becomes part of the system prompt sent to the LLM. A well-crafted backstory guides the model's tone, depth of analysis, and behavior. Compare:
- *"You are a helpful assistant"* → Generic, unfocused responses
- *"You are a skeptical MIT-trained fact-checker with 15 years experience catching misinformation..."* → Focused, critical, thorough responses

---


In [ ]:
# ============================================================
# SECTION 5: AGENT DEFINITIONS
#
# DESIGN PRINCIPLE: Each agent has a single, well-defined
# responsibility (Single Responsibility Principle).
# This makes the system modular, debuggable, and extensible.
# ============================================================
from crewai import Agent


# ── AGENT 1: THE RESEARCHER ─────────────────────────────────
# Responsibility: Gather raw, factual data using live web search.
# Only this agent has access to the search tool.
researcher = Agent(
    role="Senior Technology Researcher",
    goal=(
        "Conduct thorough research on the assigned topic using web search. "
        "Find the latest trends, key statistics, notable companies, and expert "
        "opinions. Prioritize recent sources (within the last 12 months). "
        "Always cite the source URL for every major claim you include."
    ),
    backstory=(
        "You are a veteran technology journalist with 12 years of experience "
        "covering Silicon Valley for publications like Wired and MIT Technology Review. "
        "You have a talent for finding the most credible, cutting-edge information "
        "and distinguishing signal from noise. You are meticulous about sourcing "
        "and believe that every claim needs evidence. You never speculate — "
        "if you can't find it via search, you say so."
    ),
    tools=[search_tool],   # ← ONLY the researcher gets search access
    llm=llm,
    verbose=True,           # Shows the full Thought/Action/Observation loop
    max_iter=8,             # ← IMPORTANT: See the Loop Problem note below!
    allow_delegation=False  # This agent does not sub-delegate tasks
)


# ── AGENT 2: THE TECHNICAL WRITER ───────────────────────────
# Responsibility: Transform raw research notes into a structured,
# polished technical report. No search access — works only with
# the context passed from the Researcher.
technical_writer = Agent(
    role="Principal Technical Writer",
    goal=(
        "Transform raw research notes into a clear, well-structured technical report. "
        "The report must include: an Executive Summary, 3-5 Key Trends with supporting "
        "evidence, a section on Challenges & Limitations, and a Future Outlook. "
        "Use only information explicitly provided in the research context — "
        "do not add facts from your training data."
    ),
    backstory=(
        "You are a technical writer who has authored whitepapers for Google, Microsoft, "
        "and several Y Combinator startups. You excel at making complex topics "
        "accessible to both technical and business audiences. You follow a strict "
        "discipline: you ONLY write about what is in the research notes provided to you. "
        "You never invent examples, statistics, or quotes — you flag gaps instead."
    ),
    tools=[],              # No tools — works with text context only
    llm=llm,
    verbose=True,
    max_iter=5,
    allow_delegation=False
)


# ── AGENT 3: THE FACT-CHECKER / REVIEWER ────────────────────
# Responsibility: Audit the Writer's report for hallucinations,
# unsupported claims, logical errors, and technical inaccuracies.
# This is our HALLUCINATION MITIGATION layer.
fact_checker = Agent(
    role="Chief Fact-Checker and AI Safety Auditor",
    goal=(
        "Rigorously audit the technical report for: (1) factual claims not supported "
        "by the provided research, (2) potential hallucinations or invented statistics, "
        "(3) logical inconsistencies, (4) overstated or misleading conclusions. "
        "Produce a final VERIFIED report with a clearly marked audit section listing "
        "any issues found and corrections made. Rate overall confidence: HIGH/MEDIUM/LOW."
    ),
    backstory=(
        "You are a former academic peer-reviewer with a PhD in Computer Science and "
        "a deep background in AI safety research. You are professionally skeptical — "
        "you trust nothing without a source. You have an encyclopedic knowledge of "
        "common LLM failure modes: confabulation, date errors, fake citations, and "
        "statistical distortions. Your job is to be the last line of defense before "
        "this report reaches a decision-maker. You are thorough, methodical, and "
        "you never let a suspicious claim slide without flagging it."
    ),
    tools=[],              # Works with text context only — no external search
    llm=llm,
    verbose=True,
    max_iter=5,
    allow_delegation=False
)

print(" All 3 agents defined successfully!")
print("\n   Agent Roster:")
print(f"    Agent 1: {researcher.role}")
print(f"    Agent 2: {technical_writer.role}")
print(f"    Agent 3: {fact_checker.role}")

---

###  What is an Agentic Loop?

Imagine the Researcher searches for `"AI trends 2025"` and gets no good results. A naive agent will reason:

```
THOUGHT: I didn't find good results. I should search again.
ACTION:  search("AI trends 2025")       ← Same query!
OBSERVATION: [Same poor results]
THOUGHT: I didn't find good results. I should search again.
ACTION:  search("AI trends 2025")       ← Same query again!
... [repeats 50 times, burning tokens and money]
```

This is called **"The Agentic Loop Problem"** — the agent is stuck doing the same thing and expecting different results.

###  How CrewAI Handles It

The `max_iter` parameter we set above is the **emergency brake**:

```python
researcher = Agent(
    ...  
    max_iter=8,  # ← After 8 Thought/Action cycles, STOP and return best answer
)
```

When `max_iter` is hit, CrewAI forces the agent to return its best answer so far.

###  When You Need LangGraph Instead

CrewAI's `max_iter` is a blunt instrument — it just stops the agent. For production systems with **complex branching logic**, you need **LangGraph**:

| Scenario | CrewAI | LangGraph |
|----------|--------|----------|
| Linear pipeline (A→B→C) |  Perfect fit | Overkill |
| "If no results, try different strategy" |  Limited |  Native support |
| "Go back two steps and retry" |  Not supported |  State machine handles it |
| Human-in-the-loop approval |  Basic |  First-class feature |



---
## Section 6: Defining Tasks

In CrewAI, **Tasks** are the actual work items. The distinction from Agents:
- **Agent** = Who does the work + their personality & tools
- **Task** = What specific work needs to be done + what the output should look like

### Key Task Parameters:
- **`description`** — Detailed instructions for this specific task
- **`expected_output`** — Tells the agent exactly what format/content to produce
- **`agent`** — Which agent is responsible for this task
- **`context`** — List of tasks whose outputs will be passed as context (creates dependencies)

The **`context`** parameter is what creates the **pipeline effect** — Task 2 receives Task 1's output as input.

In [ ]:
# ============================================================
# SECTION 6: TASK DEFINITIONS
#
# Tasks are defined separately from agents.
# The 'context' parameter creates data dependencies between tasks:
#   Task 1 → Task 2 (Writer receives Researcher's output)
#   Task 1 + Task 2 → Task 3 (Reviewer receives both)
# ============================================================
from crewai import Task

# --- Define the research topic ---
# Change this to explore any topic!
RESEARCH_TOPIC = "Agentic AI systems and autonomous AI workflows in 2025"


# ── TASK 1: RESEARCH ────────────────────────────────────────
research_task = Task(
    description=(
        f"Research the following topic thoroughly: '{RESEARCH_TOPIC}'.\n\n"
        "Your research should cover:\n"
        "1. The current state and latest developments (focus on last 12 months)\n"
        "2. Key market players, companies, or research groups driving this space\n"
        "3. Specific metrics, statistics, or benchmarks (with sources)\n"
        "4. Major challenges or limitations currently being faced\n"
        "5. Expert opinions or analyst forecasts\n\n"
        "Conduct at least 3 different searches with varied queries to ensure "
        "comprehensive coverage. For each key fact, note its source URL."
    ),
    expected_output=(
        "A comprehensive research briefing document containing:\n"
        "- A bullet-point summary of 10-15 key findings\n"
        "- Specific statistics and data points with source citations\n"
        "- Names of key companies, researchers, or products mentioned\n"
        "- A list of any information gaps (things you searched for but couldn't find)\n"
        "- All source URLs clearly labeled"
    ),
    agent=researcher,
    # No 'context' needed here — this is the first task
)


# ── TASK 2: WRITING ─────────────────────────────────────────
writing_task = Task(
    description=(
        "Using ONLY the research briefing provided in your context (do not use "
        "outside knowledge), write a professional technical report on the topic.\n\n"
        "The report MUST follow this exact structure:\n"
        "1. **Executive Summary** (2-3 paragraphs, high-level overview)\n"
        "2. **Key Trends & Developments** (3-5 trends, each with evidence from research)\n"
        "3. **Key Players & Ecosystem** (companies, tools, researchers)\n"
        "4. **Challenges & Limitations** (honest assessment of barriers)\n"
        "5. **Future Outlook** (what comes next, based on evidence)\n"
        "6. **Sources** (list all URLs from the research context)\n\n"
        "CRITICAL: If a claim isn't in the research notes, do NOT include it. "
        "Write [DATA NOT AVAILABLE] for any section you can't support with evidence."
    ),
    expected_output=(
        "A complete technical report in Markdown format, minimum 600 words, "
        "with all 6 required sections clearly headed. Every significant claim "
        "should be traceable to the research notes. The tone should be professional "
        "and suitable for a senior technical audience."
    ),
    agent=technical_writer,
    context=[research_task],  # ← Writer receives Researcher's output
)


# ── TASK 3: FACT-CHECKING ───────────────────────────────────
review_task = Task(
    description=(
        "You will receive two documents: (1) the original research briefing and "
        "(2) the technical report written by the writer.\n\n"
        "Your job is to cross-reference every claim in the report against the "
        "research briefing. Specifically:\n"
        "1. Identify any facts in the report NOT found in the research briefing "
        "   (potential hallucinations or training data contamination)\n"
        "2. Flag any statistics or numbers that differ from the research notes\n"
        "3. Identify any overstated conclusions beyond what evidence supports\n"
        "4. Check for logical inconsistencies within the report itself\n"
        "5. Verify that all URLs cited actually appeared in the research notes\n\n"
        "Then produce a final, corrected version of the report with your findings."
    ),
    expected_output=(
        "A final document containing two sections:\n\n"
        "## AUDIT REPORT\n"
        "- Overall Confidence Rating: HIGH / MEDIUM / LOW\n"
        "- List of issues found (or 'No issues found' if clean)\n"
        "- For each issue: [ISSUE TYPE] Description → Correction made\n\n"
        "## VERIFIED TECHNICAL REPORT\n"
        "The corrected, final version of the technical report with all issues "
        "resolved. Unchanged sections should be reproduced as-is."
    ),
    agent=fact_checker,
    context=[research_task, writing_task],  # ← Reviewer gets BOTH previous outputs
)


print(" All 3 tasks defined!")
print(f"\n    Research Topic: '{RESEARCH_TOPIC}'")
print("\n   Task Pipeline:")
print("   [Task 1: Research] → [Task 2: Writing] → [Task 3: Fact-Check]")

---
## Section 7: Assembling the Crew

The `Crew` object is the **orchestrator** — it manages the execution order, passes context between tasks, and handles the overall lifecycle.

### `Process.sequential` vs `Process.hierarchical`

- **`sequential`** — Tasks run one after another in the order listed. Output of Task N becomes context for Task N+1. **Use this for linear pipelines.**
- **`hierarchical`** — A 'manager' agent dynamically assigns tasks to workers. More flexible but requires careful setup. **Use for complex, adaptive workflows.**

We use `sequential` here because our pipeline has a natural linear order: Research → Write → Review.

### `memory=True`
When enabled, CrewAI gives agents access to a shared memory store. Agents can recall information from earlier in the same run. This is useful for long-running tasks where context window limits become a concern.

In [ ]:
# ============================================================
# SECTION 7: CREW ASSEMBLY
# The Crew object ties everything together.
# Order matters in 'agents' and 'tasks' lists for sequential process.
# ============================================================
from crewai import Crew, Process

research_crew = Crew(
    agents=[
        researcher,         # Agent 1 — executes Task 1
        technical_writer,   # Agent 2 — executes Task 2
        fact_checker,       # Agent 3 — executes Task 3
    ],
    tasks=[
        research_task,      # Step 1: Gather information
        writing_task,       # Step 2: Write the report
        review_task,        # Step 3: Audit the report
    ],
    process=Process.sequential,  # Run tasks in order, passing outputs as context
    verbose=True,                # Show crew-level orchestration logs
    memory=False,                # Set True to enable cross-task memory (uses embeddings)
    max_rpm=10,                  # Rate limit: max 10 LLM requests per minute
                                 # Prevents hitting Google API rate limits
)

print(" Crew assembled and ready!")
print("\n   Configuration Summary:")
print(f"   Process Type : Sequential")
print(f"   Agents       : {len(research_crew.agents)}")
print(f"   Tasks        : {len(research_crew.tasks)}")
print(f"   Verbose      : True")
print(f"   Max RPM      : 10")
print("\n   Ready to run! Execute the next cell to start the crew.")

---
##  Section 8: Running the Crew

This is the main execution cell. When you run it:
1. The Researcher will start searching the web (you'll see search queries and results)
2. The Writer will receive the research and draft the report
3. The Fact-Checker will audit and produce the final verified report

**Expected runtime:** 2-5 minutes depending on search response times.

**What to watch for in the verbose output:**
- `> Entering new AgentExecutor chain...` — An agent has started its ReAct loop
- `Thought:` — The agent's internal reasoning
- `Action: Search the internet` — Tool being called
- `Observation:` — Tool result being fed back to the agent
- `> Finished chain.` — Agent completed its task

In [ ]:
# ============================================================
# SECTION 8: EXECUTION
# crew.kickoff() starts the entire pipeline.
# The result object contains the final task's output.
# ============================================================
print(" Starting the Research Crew...")
print(f"   Topic: '{RESEARCH_TOPIC}'")
print("=" * 70)
print()

# --- Run the crew ---
# This is blocking — it will run until all tasks complete or an error occurs.
result = research_crew.kickoff()

print()
print("=" * 70)
print(" Crew completed successfully!")

---
## Section 9: Results & Output

The `result` object contains the final output from the last task (the Fact-Checker's verified report).
We also render it as Markdown for clean display.

In [ ]:
# ============================================================
# SECTION 9: DISPLAYING RESULTS
# ============================================================
from IPython.display import Markdown, display

print(" FINAL OUTPUT (Raw Text):")
print("=" * 70)
print(result.raw)  # .raw gives the plain text output
print("=" * 70)

print("\n FINAL OUTPUT (Rendered Markdown):")
display(Markdown(result.raw))

---
##  Section 10: Token Usage & Cost Management


###  Key Metrics to Track:
- **`prompt_tokens`** — Tokens sent TO the model (your inputs, context, history)
- **`completion_tokens`** — Tokens generated BY the model (the responses)
- **`total_tokens`** — The sum (what you're billed for)

###  Token Optimization Strategies:
1. **Shorter system prompts** — Every agent's backstory is sent with EVERY call
2. **Summarize context** — Instead of passing full previous outputs, summarize them
3. **Lower `max_iter`** — Fewer retry cycles = fewer tokens burned
4. **Choose the right model** — Flash is ~10x cheaper than Pro for the same task count
5. **Batch similar tasks** — One call for 5 similar items vs 5 separate calls

In [ ]:
# ============================================================
# SECTION 10: TOKEN USAGE & COST ANALYSIS
#
#  INSTRUCTOR NOTE:
# This is the 'bill' for the AI work just done.
# Walk students through this section to make Token Burn tangible.
# ============================================================

print(" TOKEN USAGE REPORT")
print("=" * 50)

# --- CrewAI's built-in usage metrics ---
usage = research_crew.usage_metrics
print(f"\n Raw Usage Metrics Object:")
print(f"   {usage}")

# --- Extract individual metrics ---
# Note: Attribute names may vary by CrewAI version
# Common attributes: prompt_tokens, completion_tokens, total_tokens
print(f"\n Breakdown:")

try:
    prompt_tokens = usage.prompt_tokens
    completion_tokens = usage.completion_tokens
    total_tokens = usage.total_tokens
    successful_requests = getattr(usage, 'successful_requests', 'N/A')

    print(f"   Prompt Tokens     : {prompt_tokens:,}   (input — context sent to model)")
    print(f"   Completion Tokens : {completion_tokens:,}   (output — tokens generated)")
    print(f"   Total Tokens      : {total_tokens:,}   (what you're billed for)")
    print(f"   API Calls Made    : {successful_requests}")

    # --- Cost Estimation ---
    # Gemini 1.5 Flash pricing (as of 2025, verify at ai.google.dev/pricing)
    # Input:  $0.075 per 1M tokens (prompts ≤128k)
    # Output: $0.30  per 1M tokens (completions ≤128k)
    input_cost_per_million = 0.075
    output_cost_per_million = 0.30

    estimated_input_cost = (prompt_tokens / 1_000_000) * input_cost_per_million
    estimated_output_cost = (completion_tokens / 1_000_000) * output_cost_per_million
    estimated_total_cost = estimated_input_cost + estimated_output_cost

    print(f"\n Estimated Cost (Gemini 2.5 Flash rates):")
    print(f"   Input  cost : ${estimated_input_cost:.6f}")
    print(f"   Output cost : ${estimated_output_cost:.6f}")
    print(f"   TOTAL cost  : ${estimated_total_cost:.6f}")

    # --- Scale projection ---
    print(f"\n Production Scale Projections:")
    for scale in [100, 1_000, 10_000]:
        daily_cost = estimated_total_cost * scale
        print(f"   {scale:,} requests/day → ~${daily_cost:.2f}/day (~${daily_cost*30:.0f}/month)")

except AttributeError as e:
    # Fallback if attribute names differ in the installed CrewAI version
    print(f"     Could not parse individual metrics: {e}")
    print(f"   Full metrics object: {usage}")
    print(f"   → Check your CrewAI version for the correct attribute names.")

print()
print("=" * 50)
print("\n Key Insight: Prompt tokens dominate because each LLM call")
print("   re-sends the full conversation history (LLMs are stateless).")
print("   As agent iterations increase, costs grow non-linearly.")

---
## Section 11: Inspecting Individual Task Outputs

For debugging and teaching, it's useful to inspect what each agent produced separately — not just the final output.

In [ ]:
# ============================================================
# SECTION 11: INDIVIDUAL TASK OUTPUT INSPECTION
# Useful for debugging — see exactly what each agent produced.
# ============================================================
from IPython.display import Markdown, display

task_outputs = [
    (" Task 1: Research Output", research_task),
    (" Task 2: Technical Report (Pre-Review)", writing_task),
    (" Task 3: Fact-Checked Final Report", review_task),
]

for title, task in task_outputs:
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)
    if task.output:
        display(Markdown(task.output.raw))
    else:
        print("  [No output captured — task may not have run yet]")

---
##  Section 12: Customization & Experimentation Guide

Now that you've run the crew once, here are experiments to try:

###  Experiment 1: Change the Research Topic
```python
# In Section 6, change:
RESEARCH_TOPIC = "Quantum computing breakthroughs in 2025"
# Or:
RESEARCH_TOPIC = "The impact of AI on software engineering jobs"
```

###  Experiment 2: Provoke a Hallucination
```python
# Remove the Fact-Checker from the crew:
research_crew = Crew(
    agents=[researcher, technical_writer],  # No fact_checker!
    tasks=[research_task, writing_task],    # No review_task!
    ...
)
# Compare the output — you'll often see invented statistics and fake citations
```

###  Experiment 3: Trigger the Loop Problem
```python
# Set max_iter very low to see the fallback behavior:
researcher = Agent(
    ...
    max_iter=2,  # Force early termination
)
# The agent will return incomplete research and you'll see the quality drop
```

###  Experiment 4: Temperature Effects
```python
# Try temperature=0.0 for maximum determinism:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.0)
# Then try temperature=0.9 for more creative (but less accurate) output
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.9)
```

---
##  Further Reading & Next Steps

| Resource | Description |
|----------|------------|
| [CrewAI Docs](https://docs.crewai.com) | Official documentation |
| [LangGraph](https://langchain-ai.github.io/langgraph/) | State machine framework for complex agent flows |
| [LangSmith](https://smith.langchain.com) | Production observability & tracing for LLM apps |
| [Gemini API Pricing](https://ai.google.dev/pricing) | Always check current rates |
| [Serper.dev Dashboard](https://serper.dev) | Monitor search API credits |

###  Learning Path After This Workshop:
1. **This notebook** → Linear multi-agent pipelines (CrewAI)
2. **Next** → Conditional branching agents (LangGraph)
3. **Then** → Production observability (LangSmith tracing)
4. **Advanced** → Human-in-the-loop approval workflows
5. **Expert** → Custom tool development & API integrations

---
